In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"

raw = pd.read_csv(PROC / "engineered_features_raw.csv").merge(
    pd.read_csv(PROC / "household_split.csv"), on="household_id")
FEAT = [c for c in raw.select_dtypes("number").columns if c != "month"]
train, test = raw[raw.split == "train"], raw[raw.split == "test"]

clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)
clf.fit(train[FEAT], train.true_state)
print("held-out exact accuracy:", round((clf.predict(test[FEAT]) == test.true_state).mean(), 3))
print("states:", list(clf.classes_))

held-out exact accuracy: 0.799
states: ['Critical', 'Stable', 'Strained']


In [2]:
CATS = ["housing", "food_groceries", "transport", "utilities", "airtime_data",
        "mobile_money_transfers", "loan_repayment", "school_fees", "entertainment", "savings"]
INCOME = ["income_primary", "income_secondary", "informal_cash_income"]

def build_features(m):
    """Recompute every engineered feature for one household's months, as 07_feature_engineering.py does."""
    m = m.sort_values("month").copy()
    m["total_expenditure"] = m[CATS].sum(axis=1)
    m["total_income"] = m[INCOME].sum(axis=1)
    exp = m["total_expenditure"].replace(0, np.nan)
    inc = m["total_income"].replace(0, np.nan)
    m["income_to_expenditure_ratio"] = (m["total_income"] / exp).fillna(0.0).clip(0.0, 3.0)
    m["zero_income_flag"] = (m["total_income"] == 0).astype(float)
    for c in CATS:
        m[f"{c}_share"] = (m[c] / exp).fillna(0.0)
    m["savings_rate"] = (m["savings"] / inc).fillna(0.0)
    m["monthly_debt_repayment"] = m["loan_repayment"]
    m["debt_service_coverage_ratio"] = (m["loan_repayment"] / inc).fillna(0.0).clip(upper=3.0)
    m["overspend_flag"] = (m["total_expenditure"] > m["total_income"]).astype(float)
    m["ratio_roll3"] = m["income_to_expenditure_ratio"].rolling(3, min_periods=1).mean()
    m["overspend_roll3"] = m["overspend_flag"].rolling(3, min_periods=1).mean()
    m["exp_roll3"] = m["total_expenditure"].rolling(3, min_periods=1).mean()
    m["exp_roll6"] = m["total_expenditure"].rolling(6, min_periods=1).mean()
    cm = m["total_expenditure"].expanding().mean()
    cs = m["total_expenditure"].expanding().std().fillna(0)
    m["spending_regularity_score"] = 1 / (1 + (cs / cm.replace(0, np.nan)).fillna(0))
    m["dependant_cost_factor"] = m["num_dependants"] / m["household_size"].replace(0, np.nan)
    return m

rebuilt = pd.concat([build_features(g) for _, g in raw.groupby("household_id")])
rebuilt = rebuilt.sort_values(["household_id", "month"]).reset_index(drop=True)
original = raw.sort_values(["household_id", "month"]).reset_index(drop=True)
gap = (rebuilt[FEAT] - original[FEAT]).abs().max().max()
print("largest difference from script 07:", gap)

largest difference from script 07: 5.820766091346741e-11


In [3]:
def score_row(row):
    p = dict(zip(clf.classes_, clf.predict_proba(row[FEAT])[0]))
    return 100 * (p["Stable"] + 0.5 * p["Strained"]), p

def verdict(s):
    return "Affordable" if s >= 70 else ("Stretch" if s >= 40 else "Not affordable")

def facts(r):
    r = r.iloc[0]
    return {"income_KES": round(r.total_income), "spending_KES": round(r.total_expenditure),
            "left_over_KES": round(r.total_income - r.total_expenditure),
            "income_to_spending": round(r.income_to_expenditure_ratio, 2),
            "debt_share_of_income": round(r.debt_service_coverage_ratio, 2),
            "savings_rate": round(r.savings_rate, 2)}

def affordability(hid, changes=None, income_change=0.0, months=3):
    """Score a household now and after a lifestyle change.
    changes: extra KES per month by category, e.g. {"housing": 15000} or {"loan_repayment": 12000}."""
    m = raw[raw.household_id == hid].sort_values("month").copy()
    after = m.copy()
    recent = after["month"] > after["month"].max() - months
    for cat, kes in (changes or {}).items():
        after.loc[recent, cat] += kes
    after.loc[recent, "income_primary"] += income_change
    b, a = build_features(m).iloc[[-1]], build_features(after).iloc[[-1]]
    sb, _ = score_row(b)
    sa, _ = score_row(a)
    return pd.DataFrame({"before": {"score": round(sb), "verdict": verdict(sb), **facts(b)},
                         "after": {"score": round(sa), "verdict": verdict(sa), **facts(a)}})

# Pick a held-out household that is Stable in its last month, with a middle income.
last = test[(test.month == 12) & (test.true_state == "Stable")].sort_values("total_income")
hid = last.iloc[len(last) // 2].household_id
inc = float(last.loc[last.household_id == hid, "total_income"].iloc[0])
print("household:", hid, "| monthly income KES", round(inc))

for label, change in [("rent up by 10% of income", {"housing": 0.10 * inc}),
                      ("rent up by 25% of income", {"housing": 0.25 * inc}),
                      ("new loan costing 30% of income", {"loan_repayment": 0.30 * inc})]:
    print("\n" + label)
    display(affordability(hid, change))

household: H063 | monthly income KES 25314

rent up by 10% of income


,before,after
score,90,72
verdict,Affordable,Affordable
income_KES,25314,25314
spending_KES,20715,23246
left_over_KES,4599,2068
income_to_spending,1.22,1.09
debt_share_of_income,0.0,0.0
savings_rate,0.07,0.07



rent up by 25% of income


,before,after
score,90,56
verdict,Affordable,Stretch
income_KES,25314,25314
spending_KES,20715,27043
left_over_KES,4599,-1729
income_to_spending,1.22,0.94
debt_share_of_income,0.0,0.0
savings_rate,0.07,0.07



new loan costing 30% of income


,before,after
score,90,55
verdict,Affordable,Stretch
income_KES,25314,25314
spending_KES,20715,28309
left_over_KES,4599,-2995
income_to_spending,1.22,0.89
debt_share_of_income,0.0,0.3
savings_rate,0.07,0.07


In [5]:
def explain(hid, changes=None, income_change=0.0):
    t = affordability(hid, changes, income_change)
    b, a = t["before"], t["after"]
    savings_now = round(b["savings_rate"] * b["income_KES"])
    shortfall = -a["left_over_KES"] if a["left_over_KES"] < 0 else 0
    final = a["verdict"]
    reasons = []
    if shortfall > savings_now:
        final = "Not affordable"
        reasons.append(f"You would be short KES {shortfall:,} a month, more than the KES {savings_now:,} you save.")
    elif shortfall > 0:
        reasons.append(f"You would be short KES {shortfall:,} a month; you could cover it only by pausing savings.")
    else:
        reasons.append(f"You would still have KES {a['left_over_KES']:,} left each month.")
    if a["debt_share_of_income"] >= 0.3:
        reasons.append(f"Loan repayments would take {a['debt_share_of_income']:.0%} of your income.")
    if a["income_to_spending"] < 1:
        reasons.append("You would be spending more than you earn.")
    return {"score": a["score"], "verdict": final, "score_before": b["score"], "reasons": reasons}

for label, change in [("rent up by 10% of income", {"housing": 0.10 * inc}),
                      ("rent up by 25% of income", {"housing": 0.25 * inc}),
                      ("new loan costing 30% of income", {"loan_repayment": 0.30 * inc})]:
    print(label, "->", explain(hid, change))
    

rent up by 10% of income -> {'score': 72, 'verdict': 'Affordable', 'score_before': 90, 'reasons': ['You would still have KES 2,068 left each month.']}
rent up by 25% of income -> {'score': 56, 'verdict': 'Stretch', 'score_before': 90, 'reasons': ['You would be short KES 1,729 a month; you could cover it only by pausing savings.', 'You would be spending more than you earn.']}
new loan costing 30% of income -> {'score': 55, 'verdict': 'Not affordable', 'score_before': 90, 'reasons': ['You would be short KES 2,995 a month, more than the KES 1,772 you save.', 'Loan repayments would take 30% of your income.', 'You would be spending more than you earn.']}


In [8]:
rows = []
for h in test.household_id.unique():
    now = raw[(raw.household_id == h) & (raw.month == 12)].iloc[0]
    inc_h = float(now.total_income)
    row = {"household": h, "state_now": now.true_state,
           "score_now": affordability(h)["before"]["score"]}
    for f in (0.10, 0.25, 0.50):
        row[f"rent+{int(f * 100)}%"] = affordability(h, {"housing": f * inc_h})["after"]["score"]
    rows.append(row)
res = pd.DataFrame(rows)

never_up = ((res["score_now"] >= res["rent+10%"]) & (res["rent+10%"] >= res["rent+25%"])
            & (res["rent+25%"] >= res["rent+50%"])).mean()
print("share of households where a bigger cost never raises the score:", round(never_up, 3))
print("\nscore today, by the household's real state:")
print(res.groupby("state_now")["score_now"].agg(["count", "mean", "min", "max"]).round(0))

share of households where a bigger cost never raises the score: 1.0

score today, by the household's real state:
           count  mean  min  max
state_now                       
Critical      27  20.0    1   54
Stable        15  66.0   17  100
Strained      18  46.0    3   98


In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

# +1: higher is better for the household, -1: higher is worse, 0: no direction imposed.
MONO = {"total_income": 1, "total_expenditure": -1, "income_to_expenditure_ratio": 1,
        "ratio_roll3": 1, "overspend_flag": -1, "overspend_roll3": -1,
        "exp_roll3": -1, "exp_roll6": -1, "savings_rate": 1,
        "debt_service_coverage_ratio": -1, "zero_income_flag": -1,
        "household_size": 0, "num_dependants": 0, "dependant_cost_factor": 0}
MFEAT = list(MONO)
cst = [MONO[f] for f in MFEAT]

def fit_binary(target):
    m = HistGradientBoostingClassifier(monotonic_cst=cst, max_iter=300, learning_rate=0.05, random_state=0)
    return m.fit(train[MFEAT], target)

p_stable_model = fit_binary(train.true_state == "Stable")      # P(Stable)
p_ok_model = fit_binary(train.true_state != "Critical")        # P(not Critical)

def probs(X):
    ps = p_stable_model.predict_proba(X[MFEAT])[:, 1]
    pok = np.maximum(p_ok_model.predict_proba(X[MFEAT])[:, 1], ps)  # "not Critical" can't be less likely than "Stable"
    return ps, pok

ps, pok = probs(test)
pred = np.where(ps >= 0.5, "Stable", np.where(pok >= 0.5, "Strained", "Critical"))
print("monotonic model, held-out exact accuracy:", round((pred == test.true_state).mean(), 3), "(Random Forest: 0.799)")

# From now on the affordability score uses the monotonic model. Same formula as before:
# 100 x (P(Stable) + half of P(Strained)) = 100 x (P(Stable) + P(not Critical)) / 2
def score_row(row):
    ps, pok = probs(row)
    return 100 * (ps[0] + pok[0]) / 2, None


monotonic model, held-out exact accuracy: 0.654 (Random Forest: 0.799)


In [10]:
def verdict(s):
    # Cut-offs halfway between the average scores of neighbouring states (Critical 20, Strained 46, Stable 66).
    return "Affordable" if s >= 56 else ("Stretch" if s >= 33 else "Not affordable")

joblib.dump({"state_model": clf, "state_features": FEAT,
             "p_stable_model": p_stable_model, "p_ok_model": p_ok_model,
             "score_features": MFEAT, "cutoffs": {"affordable": 56, "stretch": 33}},
            ROOT / "models" / "affordability_engine.joblib", compress=3)
print("saved:", (ROOT / "models" / "affordability_engine.joblib").stat().st_size // 1024, "KB")

for label, change in [("rent up by 10% of income", {"housing": 0.10 * inc}),
                      ("rent up by 25% of income", {"housing": 0.25 * inc}),
                      ("new loan costing 30% of income", {"loan_repayment": 0.30 * inc})]:
    print(label, "->", explain(hid, change))

saved: 3428 KB
rent up by 10% of income -> {'score': 54, 'verdict': 'Stretch', 'score_before': 85, 'reasons': ['You would still have KES 2,068 left each month.']}
rent up by 25% of income -> {'score': 48, 'verdict': 'Stretch', 'score_before': 85, 'reasons': ['You would be short KES 1,729 a month; you could cover it only by pausing savings.', 'You would be spending more than you earn.']}
new loan costing 30% of income -> {'score': 46, 'verdict': 'Not affordable', 'score_before': 85, 'reasons': ['You would be short KES 2,995 a month, more than the KES 1,772 you save.', 'Loan repayments would take 30% of your income.', 'You would be spending more than you earn.']}


In [11]:
import math

def one_off_purchase(hid, price, months=6, monthly_interest=0.0):
    """A phone, fridge, laptop: score paying in instalments, and how long saving up would take."""
    if monthly_interest > 0:
        r = monthly_interest
        per_month = price * r / (1 - (1 + r) ** -months)   # standard loan instalment formula
    else:
        per_month = price / months
    left = affordability(hid)["before"]["left_over_KES"]
    return {"price_KES": round(price),
            "instalment_per_month_KES": round(per_month),
            "if_paid_in_instalments": explain(hid, {"loan_repayment": per_month}),
            "months_to_save_up_first": math.ceil(price / left) if left > 0 else "not possible from current left-over"}

def savings_goal(hid, extra_per_month):
    """Saving more moves left-over money into savings; it is not a new cost, so it is not run through the score."""
    left = affordability(hid)["before"]["left_over_KES"]
    if extra_per_month <= left:
        return {"verdict": "Affordable",
                "reasons": [f"You have KES {left:,} left each month, so saving KES {extra_per_month:,} more fits."]}
    short = round(extra_per_month - left)
    return {"verdict": "Not affordable yet",
            "reasons": [f"You have KES {left:,} left each month, so you are KES {short:,} short.",
                        "You would need to cut other spending to reach this goal."]}

print("phone, KES 30,000 over 6 months:")
print(one_off_purchase(hid, 30000, months=6))
print("\nsave KES 3,000 more a month:", savings_goal(hid, 3000))
print("save KES 6,000 more a month:", savings_goal(hid, 6000))

phone, KES 30,000 over 6 months:
{'price_KES': 30000, 'instalment_per_month_KES': 5000, 'if_paid_in_instalments': {'score': 49, 'verdict': 'Stretch', 'score_before': 85, 'reasons': ['You would be short KES 401 a month; you could cover it only by pausing savings.', 'You would be spending more than you earn.']}, 'months_to_save_up_first': 7}

save KES 3,000 more a month: {'verdict': 'Affordable', 'reasons': ['You have KES 4,599 left each month, so saving KES 3,000 more fits.']}
save KES 6,000 more a month: {'verdict': 'Not affordable yet', 'reasons': ['You have KES 4,599 left each month, so you are KES 1,401 short.', 'You would need to cut other spending to reach this goal.']}


In [12]:
FLEXIBLE = ["entertainment", "transport", "airtime_data"]

def what_to_cut(hid, changes=None):
    """Smallest equal % cut to flexible spending that makes the scenario Affordable."""
    m = raw[raw.household_id == hid].sort_values("month")
    typical = m.tail(3)[FLEXIBLE].mean()
    for pct in range(5, 55, 5):
        cut = {c: -typical[c] * pct / 100 for c in FLEXIBLE}
        combined = dict(changes or {})
        for c, v in cut.items():
            combined[c] = combined.get(c, 0) + v
        r = explain(hid, combined)
        if r["verdict"] == "Affordable":
            return {"cut_each_by": f"{pct}%",
                    "monthly_cuts_KES": {c: round(-v) for c, v in cut.items() if v < 0},
                    "total_cut_KES": round(-sum(cut.values())),
                    "new_score": r["score"]}
    return {"message": "Even cutting flexible spending by half is not enough. Consider a cheaper option, a longer repayment period or saving up first."}

def cuts_for_savings(hid, extra_per_month):
    """How much flexible spending to cut to reach a savings goal."""
    left = affordability(hid)["before"]["left_over_KES"]
    short = extra_per_month - left
    if short <= 0:
        return {"message": "No cuts needed."}
    typical = raw[raw.household_id == hid].sort_values("month").tail(3)[FLEXIBLE].mean()
    if short > typical.sum() * 0.5:
        return {"message": f"You would need to cut KES {round(short):,} a month, more than half your flexible spending. Try a smaller goal."}
    share = typical / typical.sum()
    return {"monthly_cuts_KES": {c: round(short * share[c]) for c in FLEXIBLE}, "total_cut_KES": round(short)}

print("rent +25% of income:", what_to_cut(hid, {"housing": 0.25 * inc}))
print("loan of 30% of income:", what_to_cut(hid, {"loan_repayment": 0.30 * inc}))
print("phone in instalments:", what_to_cut(hid, {"loan_repayment": 5000}))
print("save 6,000 more:", cuts_for_savings(hid, 6000))

rent +25% of income: {'message': 'Even cutting flexible spending by half is not enough. Consider a cheaper option, a longer repayment period or saving up first.'}
loan of 30% of income: {'message': 'Even cutting flexible spending by half is not enough. Consider a cheaper option, a longer repayment period or saving up first.'}
phone in instalments: {'cut_each_by': '50%', 'monthly_cuts_KES': {'entertainment': 591, 'transport': 1582, 'airtime_data': 591}, 'total_cut_KES': 2764, 'new_score': 57}
save 6,000 more: {'monthly_cuts_KES': {'entertainment': 299, 'transport': 802, 'airtime_data': 300}, 'total_cut_KES': 1401}


In [13]:
def what_to_cut(hid, changes=None):
    """Smallest equal % cut to flexible spending to (1) stop being short, (2) reach Affordable."""
    m = raw[raw.household_id == hid].sort_values("month")
    typical = m.tail(3)[FLEXIBLE].mean()
    break_even, comfortable = None, None
    for pct in range(5, 55, 5):
        cut = {c: -typical[c] * pct / 100 for c in FLEXIBLE}
        combined = dict(changes or {})
        for c, v in cut.items():
            combined[c] = combined.get(c, 0) + v
        left = affordability(hid, combined)["after"]["left_over_KES"]
        r = explain(hid, combined)
        plan = {"cut_each_by": f"{pct}%", "monthly_cuts_KES": {c: round(-v) for c, v in cut.items()},
                "total_cut_KES": round(-sum(cut.values())), "left_over_after_KES": left,
                "score": r["score"], "verdict": r["verdict"]}
        if break_even is None and left >= 0:
            break_even = plan
        if comfortable is None and r["verdict"] == "Affordable":
            comfortable = plan
        if break_even and comfortable:
            break
    return {"to_break_even": break_even or "not reachable by cutting flexible spending by up to 50%",
            "to_be_comfortable": comfortable or "not reachable by cutting flexible spending by up to 50%"}

for label, change in [("rent +25% of income", {"housing": 0.25 * inc}),
                      ("loan of 30% of income", {"loan_repayment": 0.30 * inc}),
                      ("phone in instalments", {"loan_repayment": 5000})]:
    print("\n" + label)
    for level, plan in what_to_cut(hid, change).items():
        print(" ", level, "->", plan)


rent +25% of income
  to_break_even -> {'cut_each_by': '35%', 'monthly_cuts_KES': {'entertainment': 414, 'transport': 1107, 'airtime_data': 414}, 'total_cut_KES': 1935, 'left_over_after_KES': 205, 'score': 50, 'verdict': 'Stretch'}
  to_be_comfortable -> not reachable by cutting flexible spending by up to 50%

loan of 30% of income
  to_break_even -> not reachable by cutting flexible spending by up to 50%
  to_be_comfortable -> not reachable by cutting flexible spending by up to 50%

phone in instalments
  to_break_even -> {'cut_each_by': '10%', 'monthly_cuts_KES': {'entertainment': 118, 'transport': 316, 'airtime_data': 118}, 'total_cut_KES': 553, 'left_over_after_KES': 152, 'score': 50, 'verdict': 'Stretch'}
  to_be_comfortable -> {'cut_each_by': '50%', 'monthly_cuts_KES': {'entertainment': 591, 'transport': 1582, 'airtime_data': 591}, 'total_cut_KES': 2764, 'left_over_after_KES': 2363, 'score': 57, 'verdict': 'Affordable'}


In [14]:
# A typical student month (KES): allowance + HELB spread monthly; lives alone; no dependants.
student = {"housing": 4000, "food_groceries": 4500, "transport": 1500, "utilities": 300,
           "airtime_data": 1000, "mobile_money_transfers": 200, "loan_repayment": 0,
           "school_fees": 0, "entertainment": 800, "savings": 300,
           "income_primary": 12000, "income_secondary": 0, "informal_cash_income": 0}

def profile_months(p, n=12, household_size=1, num_dependants=0):
    m = pd.DataFrame([{**p, "month": i + 1} for i in range(n)])
    m["household_id"], m["household_size"], m["num_dependants"] = "STUDENT", household_size, num_dependants
    return m

def score_profile(m, changes=None, months=3):
    after = m.copy()
    recent = after["month"] > after["month"].max() - months
    for cat, kes in (changes or {}).items():
        after.loc[recent, cat] += kes
    f = build_features(after).iloc[[-1]]
    s, _ = score_row(f)
    r = f.iloc[0]
    return {"score": round(s), "verdict": verdict(s),
            "left_over_KES": round(r.total_income - r.total_expenditure),
            "state_now": clf.predict(f[FEAT])[0]}

m = profile_months(student)
print("student as is:", score_profile(m))
print("move off-campus (+4,000 rent):", score_profile(m, {"housing": 4000}))
print("laptop KES 45,000 over 12 months:", score_profile(m, {"loan_repayment": 3750}))

# The lump-sum trap: same money, but HELB arrives every 4 months instead of spread out.
lumpy = profile_months({**student, "income_primary": 5000})
lumpy.loc[lumpy.month.isin([1, 5, 9]), "informal_cash_income"] = 28000
print("same student, lump-sum HELB, in a month with no HELB:", score_profile(lumpy.iloc[:8]))

student as is: {'score': 47, 'verdict': 'Stretch', 'left_over_KES': -600, 'state_now': 'Strained'}
move off-campus (+4,000 rent): {'score': 12, 'verdict': 'Not affordable', 'left_over_KES': -4600, 'state_now': 'Strained'}
laptop KES 45,000 over 12 months: {'score': 12, 'verdict': 'Not affordable', 'left_over_KES': -4350, 'state_now': 'Strained'}
same student, lump-sum HELB, in a month with no HELB: {'score': 41, 'verdict': 'Stretch', 'left_over_KES': -7600, 'state_now': 'Strained'}
